In [ ]:
# =========================
# MODIFIED FOR SAFE EMAILS – BASED ON YOUR ORIGINAL
# =========================

from google.colab import drive
drive.mount('/content/drive')

# Install once (same as before)
!pip install -q --upgrade datasets pandas

import pandas as pd
from datasets import Dataset
import os

# ───── YOUR PATHS (change only if needed) ─────
DATASET_PATH = "/content/drive/MyDrive/Realsafe.xlsx"   # your file (assuming it has both phishing and safe)
OUTPUT_DIR   = "/content/drive/MyDrive/SafeEmailProject"    # new folder for safe datasets
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ───── Load your Excel file exactly as it is ─────
df = pd.read_excel(DATASET_PATH)

# Keep only safe rows (label == 0) – adjust if your safe label is different
df = df[df['label'] == 0].copy()

print(f"Loaded and filtered → {len(df)} safe emails")

# ───── One single common prompt (updated for safe emails) ─────
COMMON_PROMPT = "Generate a realistic legitimate email."

# ───── Convert every row to Vicuna chat format ─────
chat_data_strings = []
for _, row in df.iterrows():
    subject = str(row['subject']) if pd.notna(row['subject']) else "Important Update"
    body    = str(row['body'])    if pd.notna(row['body'])    else "Please review the attached information."

    assistant_reply = f"Subject: {subject}\n\n{body}"

    formatted = f"USER: {COMMON_PROMPT}\nASSISTANT: {assistant_reply}"

    chat_data_strings.append(formatted)

# ───── Create HuggingFace Dataset & split ─────
dataset = Dataset.from_dict({"text": chat_data_strings})
split   = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split["train"]
eval_dataset  = split["test"]

# ───── Save to Drive (so you never lose them) ─────
train_dataset.save_to_disk(f"{OUTPUT_DIR}/train_dataset")
eval_dataset.save_to_disk(f"{OUTPUT_DIR}/eval_dataset")

print("DONE!")
print(f"Train: {len(train_dataset)} examples")
print(f"Eval : {len(eval_dataset)} examples")
print(f"Saved to → {OUTPUT_DIR}")

# Show one example so you can see it's correct
print("\n--- EXAMPLE TRAINING ENTRY ---")
print(train_dataset[0]["text"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded and filtered → 1999 safe emails


Saving the dataset (0/1 shards):   0%|          | 0/1799 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

DONE!
Train: 1799 examples
Eval : 200 examples
Saved to → /content/drive/MyDrive/SafeEmailProject

--- EXAMPLE TRAINING ENTRY ---
USER: Generate a realistic legitimate email.
ASSISTANT: Subject: request for information

Dear Colleagues,
The National Terminology Services (NTS) of South Africa is seeking a terminology management system capable of accommodating all 11 official languages. Please note that some African languages have special diacritics not yet available in commercial software.
Attached you will find the RFI from the NTS. The detailed user requirement specification will be emailed to interested parties upon request.
Please take note of the closing date: 29 May. Your assistance in forwarding this information is greatly appreciated.
Yours sincerely,
Ms. Milde Jordan-Weiss National Terminology Services Department of Arts, Culture, Science and Technology Private Bag X8940001 Pretoria Republic of South Africa Tel: +27 12 314-6165 Fax: +27 12 325-4943


In [ ]:
# =========================
# WORKING BLOCK 2: QLoRA TRAINING FOR VICUNA-7B-V1.5 (JANUARY 2026 COLAB FIX)
# Uses PyTorch NIGHTLY – it includes the torch.load fix (version shows as 2.6.0.dev...)
# This bypasses the CVE-2025-32434 check successfully
# =========================

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install PyTorch NIGHTLY (critical – includes the vulnerability fix)
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu121

# Restart runtime automatically after install (required for torch update)
# import os
# os.kill(os.getpid(), 9)

# (After restart, run from here down)

# Install latest libraries
!pip install -q --upgrade \
    transformers \
    peft \
    bitsandbytes \
    accelerate \
    trl \
    datasets

import torch
print(f"PyTorch version: {torch.__version__}")  # Should show 2.6.0.dev... or higher nightly

import math
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, TrainerCallback, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_from_disk

# ───── PATHS ─────
OUTPUT_DIR = "/content/drive/MyDrive/SafeEmailProject"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints"
FINAL_MODEL_DIR = f"{OUTPUT_DIR}/final_model"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# ───── Load datasets ─────
train_dataset = load_from_disk(f"{OUTPUT_DIR}/train_dataset")
eval_dataset  = load_from_disk(f"{OUTPUT_DIR}/eval_dataset")

print(f"Loaded datasets → Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

# ───── Load tokenizer and model (4-bit) ─────
model_name = "arya555/vicuna-7b-v1.5-hf"  # Direct safetensors conversion of the original lmsys/vicuna-7b-v1.5

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 # Changed to bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 # Changed to bfloat16
)

model = prepare_model_for_kbit_training(model) # Removed use_reentrant=False

lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# ───── Training arguments ─────
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, # Changed from 8 to 4
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    weight_decay=0.01,
    bf16=True, # Changed from fp16=True to bf16=True for bfloat16 training
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="tensorboard", # Changed from "none" to "tensorboard"
    save_total_limit=5,
)

# ───── Perplexity callback ─────
class PerplexityCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        if "eval_loss" in metrics:
            perplexity = math.exp(metrics["eval_loss"])
            print(f"\n=== Step {state.global_step} === Validation Loss: {metrics['eval_loss']:.4f} | Perplexity: {perplexity:.2f}\n")

# ───── SFTTrainer ─────
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    callbacks=[PerplexityCallback()]
)

# ───── TRAIN ─────
print("Starting training... (will resume if checkpoint exists)")
trainer.train(resume_from_checkpoint=False)

# ───── Final evaluation ─────
eval_results = trainer.evaluate()
final_loss = eval_results["eval_loss"]
final_perplexity = math.exp(final_loss)

print(f"\nFINAL RESULTS:")
print(f"Validation Loss: {final_loss:.4f}")
print(f"Validation Perplexity: {final_perplexity:.2f}")

# ───── Save final model ─────
print("\nSaving final LoRA adapter + tokenizer to Drive...")
trainer.model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print(f"Training complete! Model saved to: {FINAL_MODEL_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Looking in indexes: https://download.pytorch.org/whl/nightly/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 18.2 MB/s eta 0:00:00
PyTorch version: 2.9.0+cu126
Loaded datasets → Train: 1799, Eval: 200


tokenizer_config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



=== Step 1200 === Validation Loss: 1.6645 | Perplexity: 5.28



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,1.652200,1.715839,1.678246,300515.000000,0.618823
400,1.645200,1.685884,1.654386,607007.000000,0.623070
600,1.601400,1.674992,1.657971,902783.000000,0.625398
800,1.541300,1.663539,1.653129,1198048.000000,0.627959
1000,1.469600,1.665539,1.575391,1512133.000000,0.628633
1200,1.429100,1.664498,1.566504,1815538.000000,0.628408



=== Step 1350 === Validation Loss: 1.6635 | Perplexity: 5.28


FINAL RESULTS:
Validation Loss: 1.6635
Validation Perplexity: 5.28

Saving final LoRA adapter + tokenizer to Drive...
Training complete! Model saved to: /content/drive/MyDrive/SafeEmailProject/final_model
